In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)# Inspecionar colunas

In [ ]:
from src.data_io import load_portaria_estrutura_territorial_ima

portaria_df = load_portaria_estrutura_territorial_ima()

In [ ]:
portaria_df.head()

In [ ]:
portaria_df.dtypes

In [ ]:
# ============================================================
# PADRONIZAÇÃO DE COLUNAS DA PORTARIA
# ============================================================

# Renomeia colunas da planilha da portaria para o padrão usado no projeto,
# permitindo integração direta (merge) com os demais datasets

portaria_df = portaria_df.rename(columns={
    "IBGE": "codigo_do_municipio",
    "Município": "descricao_do_municipio",
    "ESEC Vinculado": "esec",
    "CR Vinculada": "coordenadoria",
})

In [ ]:
portaria_df.head()

In [ ]:
import pandas as pd
import unicodedata


def _normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Padroniza separadores
    text = text.replace("_", " ")

    # Remove espaços duplicados
    text = " ".join(text.split())

    return text

In [ ]:
# descricao_do_municipio → minúsculo + sem acento
portaria_df["descricao_do_municipio"] = portaria_df["descricao_do_municipio"].apply(_normalize_text)


# esec → remove prefixo "ESEC" + normaliza
portaria_df["esec"] = (
    portaria_df["esec"]
    .str.replace(r"^esec\s+", "", regex=True)
    .apply(_normalize_text)
)


# coordenadoria → remove prefixo "CR" + normaliza
portaria_df["coordenadoria"] = (
    portaria_df["coordenadoria"]
    .str.replace(r"^cr\s+", "", regex=True)
    .apply(_normalize_text)
)

In [ ]:
portaria_df.head()

In [ ]:
from src.data_io import load_propriedades_processed

df = load_propriedades_processed()

In [ ]:
df.head()

In [ ]:
# Mantém apenas as colunas que também existem no portaria_df
df = df[portaria_df.columns.intersection(df.columns)]

In [ ]:
df.head()

In [ ]:
# descricao_do_municipio
df["descricao_do_municipio"] = df["descricao_do_municipio"].apply(_normalize_text)

# esec (sem remover prefixo)
df["esec"] = df["esec"].apply(_normalize_text)

# coordenadoria (sem remover prefixo)
df["coordenadoria"] = df["coordenadoria"].apply(_normalize_text)

In [ ]:
df.head()

In [ ]:
# Mantém apenas um registro por município
df = df.drop_duplicates(subset=["codigo_do_municipio"])

In [ ]:
duplicados = df["codigo_do_municipio"].value_counts()
duplicados = duplicados[duplicados > 1]

print(duplicados)
print(f"\nTotal de municípios duplicados: {duplicados.shape[0]}")

In [ ]:
# Merge
comparacao_df = df.merge(
    portaria_df,
    on="codigo_do_municipio",
    how="left",
    suffixes=("_df", "_portaria"),
)

# Flags de comparação
comparacao_df["esec_bate"] = (
    comparacao_df["esec_df"] == comparacao_df["esec_portaria"]
)

comparacao_df["coordenadoria_bate"] = (
    comparacao_df["coordenadoria_df"] == comparacao_df["coordenadoria_portaria"]
)

# Filtra onde NÃO bate
erros = comparacao_df[
    ~(comparacao_df["esec_bate"] & comparacao_df["coordenadoria_bate"])
]

# Mostra só os códigos problemáticos
print(erros["codigo_do_municipio"].unique())

# Se quiser ver detalhado
erros[[def _normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    text = " ".join(text.split())

    return text
    "codigo_do_municipio",
    "esec_df", "esec_portaria",
    "coordenadoria_df", "coordenadoria_portaria"
]]